# Torsion Scan Guardian — Thunder Compute single-molecule demo

Runs the Phase-5 active-learning demo on sulfanilamide on a Thunder Compute GPU instance, driven from VSCode (via the Thunder VSCode extension OR Remote-SSH).

**Prerequisite:** you've already followed [`Thunder_WAY.md`](../../Thunder_WAY.md) steps 1–7 to:
- Create a Thunder GPU instance
- Connect VSCode to it via SSH
- Clone this repo on the instance
- Install Python deps (`pip install -e .` + xtb)
- Pre-warm the MACE-OFF cache

This notebook is the **post-setup** part — running the actual experiment. Wall time on a T4: ~3–5 min.

## 1. Verify the environment

In [ ]:
import os, torch
print('cwd:', os.getcwd())
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
assert os.path.exists('pyproject.toml'), 'Run this notebook from the repo root (cd torsion_scan_guardian first).'

In [ ]:
# Pre-warm the MACE-OFF cache if it isn't already (idempotent).
from mace.calculators import mace_off
mace_off(model='medium', device='cuda' if torch.cuda.is_available() else 'cpu')
p = os.path.expanduser('~/.cache/mace/MACE-OFF23_medium.model')
assert os.path.exists(p), f'MACE-OFF cache missing at {p}'
print('MACE-OFF cache:', os.path.getsize(p) // 1024, 'KB')

In [ ]:
!python -m pytest -q tests/ -k 'not test_ensemble_predict_h2o'

## 2. Phase 5 demo on sulfanilamide

Same configuration as REPORT §12.5. Wall time: **~3–5 min on T4**.

If you already ran the sweep notebook and the seed dataset + fine-tune ensemble for sulfanilamide already exist, steps 2a and 2b are skipped automatically.

In [ ]:
%%bash
# Step 2a: build the seed dataset (~3 min).
if [ ! -f data/seed/sulfanilamide_seed.xyz ]; then
    python scripts/build_seed_dataset.py \
        --smiles 'Nc1ccc(S(=O)(=O)N)cc1' \
        --out data/seed/sulfanilamide_seed.xyz
else
    echo "seed dataset exists at data/seed/sulfanilamide_seed.xyz — skipping build"
fi

In [ ]:
%%bash
# Step 2b: fine-tune 3 ensemble members on GPU (~1–2 min total).
for SEED in 0 1 2 3 4; do
    CKPT=runs/finetune_sulf_medium/member_seed${SEED}/checkpoints/member_seed${SEED}_run-${SEED}.model
    if [ ! -f "$CKPT" ]; then
        MPLBACKEND=Agg python scripts/finetune_member.py \
            --seed $SEED --epochs 5 --lr 5e-4 --foundation-size medium \
            --train-file data/seed/sulfanilamide_seed.xyz \
            --out-root runs/finetune_sulf \
            --device cuda
    else
        echo "member $SEED already trained — skipping"
    fi
done
ls runs/finetune_sulf_medium/*/checkpoints/*.model

In [ ]:
%%bash
# Step 2c: run the 5-cycle AL loop with cloud acquisitions + safeguarded fine-tune.
MPLBACKEND=Agg python -m guardian.cli \
    --config config/default.yaml \
    --smiles 'Nc1ccc(S(=O)(=O)N)cc1' \
    --steps 4000 --temperature 300 --threshold 0.05 \
    --max-triggers 5 --cooldown-steps 200 \
    --cloud-size 5 --cloud-jitter 0.05 \
    --ft-regression-tol 0.10 \
    --checkpoints runs/finetune_sulf_medium/member_seed0/checkpoints/*.model \
                  runs/finetune_sulf_medium/member_seed1/checkpoints/*.model \
                  runs/finetune_sulf_medium/member_seed2/checkpoints/*.model \
    --online-finetune \
    --seed-data-file data/seed/sulfanilamide_seed.xyz \
    --finetune-epochs 2 --finetune-lr 1e-4 \
    --run-dir runs/sulf_phase5_thunder

In [ ]:
# Step 2d: analysis figures.
!MPLBACKEND=Agg python scripts/analyse_al_demo.py runs/sulf_phase5_thunder

In [ ]:
from IPython.display import Image
Image('figures/sulf_phase5_thunder_timeline.png')

## 3. Push results back to GitHub

Three options documented in [`Thunder_WAY.md`](../../Thunder_WAY.md) §9 — pick what fits your security preferences:

- **Option A — pull artifacts to local + push from local** (no creds on Thunder). Run from your local terminal:
  ```bash
  scp -P <port> -i ~/.ssh/<key> root@<host>:~/torsion_scan_guardian/figures/sulf_phase5_thunder_*.png ./figures/
  git add figures/sulf_phase5_thunder_*.png
  git commit -m "Phase 5 demo on Thunder: figures"
  git push
  ```
- **Option B — fine-grained PAT on the Thunder instance**. Generate at github.com/settings/personal-access-tokens with `Contents: Read and write` on this repo only, then:
  ```bash
  !git remote set-url origin https://Sathapana:<TOKEN>@github.com/Sathapana/torsion_scan_guardian.git
  !git add figures/sulf_phase5_thunder_*.png && git commit -m "..." && git push
  ```
  Revoke the PAT when done with the instance.
- **Option C — SSH key on Thunder**. Generate ed25519 on the instance, add to GitHub, use `git@github.com:Sathapana/torsion_scan_guardian.git` remote.

## 4. Stop the instance when done

From your **local** terminal (not the Thunder shell):
```bash
tnr stop <instance-id>
```
Forgetting this is the #1 way to be surprised by a Thunder bill. Make it a habit.